# FocusAI — Dual-Input Model Training
Trains a dual-input neural network that fuses cropped face images with
head-pose angles (yaw, pitch, roll) to classify focus state.
Images from `data/cropped/focused/` (label 0) and `data/cropped/distracted/` (label 1).

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path("../").resolve()
sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
import cv2
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight

from src.utils.config import (
    IMG_SIZE, CHANNELS, CLASSES,
    DATA_CROPPED_PATH, LANDMARK_CSV_PATH, MODEL_PATH,
)

np.random.seed(42)
tf.random.set_seed(42)

print("TensorFlow:", tf.__version__)
print("IMG_SIZE:", IMG_SIZE, "| CHANNELS:", CHANNELS, "| CLASSES:", CLASSES)


## 1 Carregamento das Imagens
Reads every `.jpg` from `data/cropped/focused/` and `data/cropped/distracted/` using OpenCV
in grayscale. Each image is resized to 224×224, normalised to [0, 1], and stored
with a `(class, filename)` composite key for later alignment with the CSV.

In [ ]:
data_dir = PROJECT_ROOT / DATA_CROPPED_PATH

# (class_name, filename) -> (normalised_image_array, label_index)
image_store: dict[tuple, tuple] = {}

for label_idx, class_name in enumerate(CLASSES):
    class_dir = data_dir / class_name
    jpg_files = sorted(class_dir.glob("*.jpg"))
    print(f"{class_name}: {len(jpg_files)} images")
    for img_path in jpg_files:
        img = cv2.imread(str(img_path), cv2.IMREAD_GRAYSCALE)
        img = cv2.resize(img, IMG_SIZE)
        # Normalise to [0, 1] and add the channel dim Keras expects
        img_norm = img.astype(np.float32) / 255.0
        img_norm = img_norm.reshape(*IMG_SIZE, CHANNELS)
        image_store[(class_name, img_path.name)] = (img_norm, label_idx)

print(f"\nTotal images loaded: {len(image_store)}")


## 2 Carregamento dos Ângulos
Reads `landmarks.csv` (columns: `filename`, `class`, `yaw`, `pitch`, `roll`, `face_detected`).
Null angles (face not detected) are replaced with the sentinel value `999.0`.

In [ ]:
csv_path = PROJECT_ROOT / LANDMARK_CSV_PATH
angles_df = pd.read_csv(csv_path)

# NaN angles mean the face was not detected in that frame -> sentinel 999.0
angles_df[["yaw", "pitch", "roll"]] = angles_df[["yaw", "pitch", "roll"]].fillna(999.0)

no_face_count = (angles_df["yaw"] == 999.0).sum()
print(f"CSV rows     : {len(angles_df)}")
print(f"No-face rows : {no_face_count} (sentinel 999.0 applied)")
angles_df.head()


## 3 Pré-processamento
Joins images and angles by `(class, filename)`, builds numpy arrays, and creates a
stratified 80/20 train/val split. Class weights compensate for the
~997 focused vs ~1339 distracted imbalance.

In [ ]:
X_images, X_angles, y_labels = [], [], []
missing = 0

for _, row in angles_df.iterrows():
    key = (row["class"], Path(row["filename"]).name)
    if key not in image_store:
        # CSV entry has no corresponding image file on disk
        missing += 1
        continue
    img_norm, label_idx = image_store[key]
    X_images.append(img_norm)
    X_angles.append([row["yaw"], row["pitch"], row["roll"]])
    y_labels.append(label_idx)

print(f"Aligned samples : {len(X_images)} | Unmatched : {missing}")

X_images = np.array(X_images, dtype=np.float32)
X_angles = np.array(X_angles, dtype=np.float32)
y = np.array(y_labels, dtype=np.int32)

# Stratified split preserves the focused/distracted ratio in both sets
X_img_train, X_img_val, X_ang_train, X_ang_val, y_train, y_val = train_test_split(
    X_images, X_angles, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Train : {len(y_train)}  |  Val : {len(y_val)}")

# Inverse-frequency weights so the minority class (focused) is not overshadowed
class_weights_arr = compute_class_weight("balanced", classes=np.unique(y_train), y=y_train)
class_weight = dict(enumerate(class_weights_arr))
print(f"Class weights : {class_weight}")


## 4 Arquitetura do Modelo
Dual-input Keras functional model: a 3-block CNN for images concatenated with
a 2-layer dense network for head-pose angles, fused into a binary sigmoid output.

In [ ]:
# --- Image branch ---
image_input = keras.Input(shape=(*IMG_SIZE, CHANNELS), name="image_input")

x = layers.Conv2D(32, 3, padding="same", activation="relu")(image_input)
x = layers.MaxPooling2D()(x)

x = layers.Conv2D(64, 3, padding="same", activation="relu")(x)
x = layers.MaxPooling2D()(x)

x = layers.Conv2D(128, 3, padding="same", activation="relu")(x)
x = layers.MaxPooling2D()(x)

x = layers.Flatten()(x)
x = layers.Dense(128, activation="relu")(x)
x = layers.Dropout(0.4)(x)

# --- Angle branch ---
angle_input = keras.Input(shape=(3,), name="angle_input")
a = layers.Dense(16, activation="relu")(angle_input)
a = layers.Dense(16, activation="relu")(a)

# --- Fusion head ---
merged = layers.Concatenate()([x, a])
out = layers.Dense(64, activation="relu")(merged)
out = layers.Dropout(0.3)(out)
out = layers.Dense(1, activation="sigmoid")(out)

model = keras.Model(
    inputs=[image_input, angle_input], outputs=out, name="focus_dual"
)
model.summary()


## 5 Compilação e Treino
Adam (lr=1e-3), binary cross-entropy loss. EarlyStopping (patience=8),
ReduceLROnPlateau (patience=4), and ModelCheckpoint save the best model.
Class weights are passed to `model.fit` to handle imbalance during training.

In [ ]:
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss="binary_crossentropy",
    metrics=["accuracy"],
)

ckpt_path = PROJECT_ROOT / MODEL_PATH
ckpt_path.parent.mkdir(parents=True, exist_ok=True)

callbacks = [
    keras.callbacks.EarlyStopping(
        monitor="val_loss", patience=8, restore_best_weights=True
    ),
    keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss", factor=0.5, patience=4, min_lr=1e-6
    ),
    keras.callbacks.ModelCheckpoint(
        filepath=str(ckpt_path), monitor="val_loss", save_best_only=True
    ),
]

history = model.fit(
    [X_img_train, X_ang_train], y_train,
    validation_data=([X_img_val, X_ang_val], y_val),
    epochs=50,
    batch_size=32,
    callbacks=callbacks,
    class_weight=class_weight,
    verbose=1,
)


## 6 Avaliação e Métricas
Prints validation loss/accuracy, plots accuracy and loss curves side by side,
and shows a scikit-learn classification report with class names.

In [ ]:
from sklearn.metrics import classification_report

loss, acc = model.evaluate([X_img_val, X_ang_val], y_val, verbose=0)
print(f"Val loss: {loss:.4f}  |  Val accuracy: {acc:.4f}")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(history.history["loss"], label="train")
ax1.plot(history.history["val_loss"], label="val")
ax1.set_title("Loss")
ax1.legend()

ax2.plot(history.history["accuracy"], label="train")
ax2.plot(history.history["val_accuracy"], label="val")
ax2.set_title("Accuracy")
ax2.legend()

plt.tight_layout()
plt.show()

y_pred = (model.predict([X_img_val, X_ang_val], verbose=0) > 0.5).astype(int).flatten()
print(classification_report(y_val, y_pred, target_names=CLASSES))


## 7 Salvamento do Modelo
Saves the model with best weights (restored by EarlyStopping) to `models/focus_model.h5`.

In [ ]:
model_out = PROJECT_ROOT / MODEL_PATH
model_out.parent.mkdir(parents=True, exist_ok=True)
model.save(str(model_out))
print("Model saved to:", model_out)
